# Electricity market price dataset generation

Process json files from REE API (https://www.ree.es/es/apidatos) into a dataframe.

In [16]:
from pathlib import Path
import json
import pandas as pd
from loguru import logger

data_path: Path = Path("/workspaces/SOLhycool/simulation/results/electricity_price_data")
output_path: Path = Path("/workspaces/SOLhycool/data/datasets")
json_files = list(data_path.glob("*.json"))

df = pd.DataFrame()
for file_idx, file in enumerate(json_files):
    data = json.load(file.open("r"))
    logger.info(f"{file_idx+1:02d} / {len(json_files)} | Processing {file.name}")
    source_ids = [data_source["type"].replace(" ", "_").replace("-", "_").lower() for data_source in data["included"]]
            
    temp_df = pd.DataFrame(data["included"][0]["attributes"]["values"])
    time = pd.to_datetime(temp_df["datetime"], utc=True)
    output = {f"Ce_{src_id}_eur_MWh": pd.DataFrame(data_src["attributes"]["values"])["value"].values for src_id, data_src in zip(source_ids, data["included"])}
    # Add columns with additional units
    for col_id, values in list(output.items()):
        output[col_id.replace("MWh", "kWh")] = values * 1e-3  # 1 MWh = 1000 kWh

    df = pd.concat([df, pd.DataFrame(output, index=time)], axis=0)

df.sort_index(inplace=True)
display(df)

logger.info(f"Resulting dataframe has {len(df)} rows")

# Save to hdf5 and csv
fname = f"electricity_price_data_{df.index[0].strftime('%Y%m%d')}_{df.index[-1].strftime('%Y%m%d')}"
df.to_hdf(output_path / f"{fname}.h5", key="data", mode="w", format="table")
df.to_csv(output_path / f"{fname}.csv")
    
logger.info(f"Saved {fname} to {output_path}")


2025-02-28 12:12:17.131 | INFO     | __main__:<module>:13 - 01 / 36 | Processing 20231201_20231231.json
2025-02-28 12:12:17.143 | INFO     | __main__:<module>:13 - 02 / 36 | Processing 20240101_20240131.json
2025-02-28 12:12:17.153 | INFO     | __main__:<module>:13 - 03 / 36 | Processing 20240801_20240831.json
2025-02-28 12:12:17.162 | INFO     | __main__:<module>:13 - 04 / 36 | Processing 20230101_20230131.json
2025-02-28 12:12:17.171 | INFO     | __main__:<module>:13 - 05 / 36 | Processing 20240301_20240331.json
2025-02-28 12:12:17.179 | INFO     | __main__:<module>:13 - 06 / 36 | Processing 20240901_20240930.json
2025-02-28 12:12:17.188 | INFO     | __main__:<module>:13 - 07 / 36 | Processing 20221101_20221130.json
2025-02-28 12:12:17.197 | INFO     | __main__:<module>:13 - 08 / 36 | Processing 20220901_20220930.json
2025-02-28 12:12:17.206 | INFO     | __main__:<module>:13 - 09 / 36 | Processing 20240701_20240731.json
2025-02-28 12:12:17.216 | INFO     | __main__:<module>:13 - 10 /

,Ce_pvpc_eur_MWh,Ce_spot_market_price_eur_MWh,Ce_pvpc_eur_kWh,Ce_spot_market_price_eur_kWh
datetime,,,,
2021-12-31 23:00:00+00:00,204.51,145.86,0.20451,0.14586
2022-01-01 00:00:00+00:00,171.35,114.90,0.17135,0.11490
2022-01-01 01:00:00+00:00,172.70,113.87,0.17270,0.11387
2022-01-01 02:00:00+00:00,156.07,97.80,0.15607,0.09780
2022-01-01 03:00:00+00:00,159.08,97.80,0.15908,0.09780
...,...,...,...,...
2024-12-31 18:00:00+00:00,274.90,159.44,0.27490,0.15944
2024-12-31 19:00:00+00:00,271.27,156.10,0.27127,0.15610
2024-12-31 20:00:00+00:00,268.41,150.85,0.26841,0.15085


2025-02-28 12:12:17.491 | INFO     | __main__:<module>:28 - Resulting dataframe has 26304 rows
2025-02-28 12:12:17.692 | INFO     | __main__:<module>:35 - Saved electricity_price_data_20211231_20241231 to /workspaces/SOLhycool/data/datasets


In [17]:
df = pd.read_hdf(output_path / "electricity_price_data_20211231_20241231.h5", index_col=0, parse_dates=True)
df


,Ce_pvpc_eur_MWh,Ce_spot_market_price_eur_MWh,Ce_pvpc_eur_kWh,Ce_spot_market_price_eur_kWh
datetime,,,,
2021-12-31 23:00:00+00:00,204.51,145.86,0.20451,0.14586
2022-01-01 00:00:00+00:00,171.35,114.90,0.17135,0.11490
2022-01-01 01:00:00+00:00,172.70,113.87,0.17270,0.11387
2022-01-01 02:00:00+00:00,156.07,97.80,0.15607,0.09780
2022-01-01 03:00:00+00:00,159.08,97.80,0.15908,0.09780
...,...,...,...,...
2024-12-31 18:00:00+00:00,274.90,159.44,0.27490,0.15944
2024-12-31 19:00:00+00:00,271.27,156.10,0.27127,0.15610
2024-12-31 20:00:00+00:00,268.41,150.85,0.26841,0.15085


In [5]:
from plotly_resampler import FigureWidgetResampler
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

df = df.resample("h").mean().ffill()

var_ids: list[str] = ["Ce_pvpc_eur_MWh", "Ce_spot_market_price_eur_MWh"]
var_units: list[str] = ["€/MWh", "€/MWh"]

# fig = make_subplots(rows=len(var_ids), cols=1, shared_xaxes=True)
# fig = go.FigureWidget()
fig = FigureWidgetResampler()

for i, (var_id, var_unit) in enumerate(zip(var_ids, var_units)):
    fig.add_trace(
        go.Scattergl(name=var_id, showlegend=True), #x=df.index, y=df[var_id], mode="lines", line=dict(width=1)), 
        hf_x=df.index, 
        hf_y=np.ascontiguousarray(df[var_id].values), 
        max_n_samples=2_000,
        # row=i + 1, col=1
    )
    fig.update_yaxes(title_text=f"({var_unit})")#, row=i + 1)

fig.update_layout(
    height=650,
    width=800,
    title="<b>Market electricity price</b>",
    title_x=0.1,
    legend_traceorder="normal",
    legend=dict(orientation="h", y=1.08, xanchor="left", x=0),
)

fig

# fig.show(
#     config=generate_plotly_config(
#         fig, figure_name=f"solhycool_env_viz_{df_env.index[0].strftime('%Y%m%d')}"
#     )
# )


FigureWidgetResampler({
    'data': [{'name': ('<b style="color:sandybrown">[R' ... 'style="color:#fc9944">~13h</i>'),
              'showlegend': True,
              'type': 'scattergl',
              'uid': 'b8bf1321-fa4c-49b5-8d72-6761ff7d4602',
              'x': array([datetime.datetime(2021, 12, 31, 23, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 1, 8, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 1, 19, 0, tzinfo=datetime.timezone.utc), ...,
                          datetime.datetime(2024, 12, 30, 19, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 31, 13, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 31, 22, 0, tzinfo=datetime.timezone.utc)],
                         shape=(2000,), dtype=object),
              'y': array([204.51, 118.96, 247.11, ..., 288.57, 166.81, 209.24], shape=(2000,))},
             {'name': 

In [25]:
from phd_visualizations import save_figure

start, end = fig.layout.xaxis.range
save_figure(fig, f"solhycool_electricity_price_{start[:10].replace('-', '')}_{end[:10].replace('-', '')}", figure_path=data_path, formats=["png", "svg"])


2025-02-17 10:30:34.572 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('/workspaces/SOLhycool/simulation/results/electricity_price_data')]/solhycool_electricity_price_20211231_20241231.png
2025-02-17 10:30:37.134 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('/workspaces/SOLhycool/simulation/results/electricity_price_data')]/solhycool_electricity_price_20211231_20241231.svg
